<a href="https://colab.research.google.com/github/Thanseeh07/Ml-lab/blob/main/experiment%205.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

categories=['alt.atheism','soc.religion.christian','comp.graphics','sci.med']
print("Loading 20 Newsgroups dataset.......")
train_data=fetch_20newsgroups(subset='train',categories=categories,remove=('headers','footers','quotes'))
test_data=fetch_20newsgroups(subset='test',categories=categories,remove=('headers','footers','quotes'))

vectorizer=CountVectorizer(stop_words='english',max_features=5000)
x_train=vectorizer.fit_transform(train_data.data).toarray()
y_train=train_data.target
x_test=vectorizer.transform(test_data.data).toarray()
y_test=test_data.target

n_classes=len(categories)
n_features=x_train.shape[1]
print(f"Train shape: {x_train.shape}, Test shape: {x_test.shape}")

class_counts = np.bincount(y_train)
p_labels = class_counts / len(y_train)


N_c = np.zeros((n_classes, n_features))
for c in range(n_classes):
    N_c[c, :] = x_train[y_train == c].sum(axis=0)
print("Word counts aggregated successfully.")
def predict_naive_bayes(X, theta, p_labels):
    """
    Predicts classes using log-probabilities to avoid underflow.
    Log-likelihood: X . log(Theta^T) + log(P(c))
    """
    with np.errstate(divide='ignore'):
        log_theta = np.log(theta)


    log_theta[np.isinf(log_theta)] = -1e9

    log_posterior = X @ log_theta.T + np.log(p_labels)
    return np.argmax(log_posterior, axis=1)
theta_mle = N_c / N_c.sum(axis=1, keepdims=True)

y_pred_mle = predict_naive_bayes(x_test, theta_mle, p_labels)
mle_accuracy = accuracy_score(y_test, y_pred_mle)
print(f"[MLE] Test Accuracy: {mle_accuracy:.4f}")
priors_to_test = {
    "Lidstone Smoothing (Alpha = 1.01)": np.ones(n_features) * 1.01,
    "Laplace Smoothing (Alpha = 2.0)": np.ones(n_features) * 2.0,
    "Strong Dirichlet Prior (Alpha = 10.0)": np.ones(n_features) * 10.0,
    "Non-Uniform Prior (Empirical Basis)": 1.0 + (x_train.sum(axis=0) / x_train.sum() * 100)
}

print("Evaluating MAP Predictions:")
print("-" * 55)
for name, alpha in priors_to_test.items():
    numerator = N_c + (alpha - 1)
    denominator = numerator.sum(axis=1, keepdims=True)
    theta_map = numerator / denominator

    y_pred_map = predict_naive_bayes(x_test, theta_map, p_labels)
    map_accuracy = accuracy_score(y_test, y_pred_map)
    print(f"{name:38} | Test Accuracy: {map_accuracy:.4f}")

Loading 20 Newsgroups dataset.......
Train shape: (2257, 5000), Test shape: (1502, 5000)
Word counts aggregated successfully.
[MLE] Test Accuracy: 0.7710
Evaluating MAP Predictions:
-------------------------------------------------------
Lidstone Smoothing (Alpha = 1.01)      | Test Accuracy: 0.8103
Laplace Smoothing (Alpha = 2.0)        | Test Accuracy: 0.8182
Strong Dirichlet Prior (Alpha = 10.0)  | Test Accuracy: 0.7863
Non-Uniform Prior (Empirical Basis)    | Test Accuracy: 0.8063
